# Minimum SAE Circuit Discovery v003: Role-Conditioned Circuits

This notebook tests a more publishable unit of analysis: semantic-role-conditioned SAE circuits.

Instead of selecting global SAE features, it selects `role x SAE feature` nodes. For IOI, a feature may matter at the indirect-object token but not at every token. The hypothesis is that role-conditioned circuits can preserve faithfulness with far fewer active nodes than v2's global feature masks.

The notebook compares v2-style global masks, static role-feature rankings, core-role-only rankings, and an optional learned continuous gate over top role-feature candidates.


## 1. Colab Setup

Run this notebook in a GPU runtime. The smoke cells are small; the full baseline comparison runs several top-K sweeps and is intended for Colab Pro+.


In [ ]:
%pip install -q "sae-lens>=6,<7" pandas matplotlib tqdm


In [ ]:
import json
import os
import random
import shutil
from collections import defaultdict
from contextlib import nullcontext
from datetime import datetime
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from sae_lens import SAE, HookedSAETransformer
from tqdm.auto import tqdm

SEED = 12345
random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook expects a CUDA GPU. In Colab, enable Runtime -> Change runtime type -> GPU.")

device = "cuda"
torch.set_grad_enabled(False)

props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")


## 2. Versioned Drive Output

All v3 artifacts go to a separate Google Drive version folder so v1/v2 results remain intact.


In [ ]:
MOUNT_DRIVE = True
PROJECT_DIR_NAME = "minimum_sae_circuit_discovery"
RUN_VERSION = "v003_role_conditioned_circuits"
TRIAL_ID = os.environ.get("TRIAL_ID") or datetime.now().strftime("trial_%Y%m%d_%H%M%S")
RUN_STARTED_AT = datetime.now().astimezone().isoformat(timespec="seconds")

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and MOUNT_DRIVE:
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_DIR_NAME
else:
    PROJECT_ROOT = Path.cwd() / f"{PROJECT_DIR_NAME}_outputs"

CACHE_DIR = PROJECT_ROOT / "cache" / RUN_VERSION
OUTPUT_DIR = PROJECT_ROOT / "runs" / RUN_VERSION / TRIAL_ID
LATEST_DIR = PROJECT_ROOT / "latest" / RUN_VERSION
for directory in (CACHE_DIR, OUTPUT_DIR, LATEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "gpt2-small"
SAE_RELEASE = "gpt2-small-res-jb"
TARGET_LAYER = 8
EXPECTED_D_SAE = 24_576

SMOKE_N_PROMPTS = 8
N_PROMPTS = 1_000
OPTIMIZATION_N_PROMPTS = 256
BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 8

GLOBAL_K_VALUES = [20, 50, 100, 200, 500, 1_000, 2_000, 5_000]
ROLE_K_VALUES = [10, 20, 50, 100, 200, 500, 1_000, 2_000, 4_000, 8_000]
SMOKE_GLOBAL_K_VALUES = [20, 100, 500]
SMOKE_ROLE_K_VALUES = [20, 100, 500]

TOP_GLOBAL_FEATURE_POOL = 1_000
LEARNED_CANDIDATE_PAIRS = 2_000
RUN_LEARNED_ROLE_MASK = True
LEARNED_LAMBDA = 0.03
LEARNED_STEPS = 200
LEARNED_LR = 0.08

STATS_CACHE_PATH = CACHE_DIR / f"role_feature_stats_layer{TARGET_LAYER}_ioi.pt"
SMOKE_STATS_CACHE_PATH = CACHE_DIR / f"smoke_role_feature_stats_layer{TARGET_LAYER}_ioi.pt"
SMOKE_RESULTS_CSV_PATH = OUTPUT_DIR / "smoke_role_conditioned_results.csv"
RESULTS_CSV_PATH = OUTPUT_DIR / "role_conditioned_results.csv"
LEARNED_TRACE_CSV_PATH = OUTPUT_DIR / "learned_role_mask_trace.csv"
SMOKE_PLOT_PATH = OUTPUT_DIR / "smoke_role_conditioned_pareto.png"
PLOT_PATH = OUTPUT_DIR / "role_conditioned_pareto.png"
MANIFEST_PATH = OUTPUT_DIR / "run_manifest.json"

ROLE_NAMES = ["bos", "subject_first", "indirect_object", "subject_repeat", "place", "object", "final_prediction", "other"]
ROLE_TO_ID = {name: idx for idx, name in enumerate(ROLE_NAMES)}
CORE_ROLE_NAMES = ["bos", "subject_first", "indirect_object", "subject_repeat", "final_prediction"]
CORE_ROLE_IDS = torch.tensor([ROLE_TO_ID[name] for name in CORE_ROLE_NAMES], dtype=torch.long)

def hook_name_for_layer(layer):
    return f"blocks.{layer}.hook_resid_pre"

def sae_id_for_layer(layer):
    return hook_name_for_layer(layer)

def write_run_manifest(status, extra=None):
    manifest = {
        "status": status,
        "run_started_at": RUN_STARTED_AT,
        "run_updated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "run_version": RUN_VERSION,
        "trial_id": TRIAL_ID,
        "seed": SEED,
        "model_name": MODEL_NAME,
        "sae_release": SAE_RELEASE,
        "target_layer": TARGET_LAYER,
        "n_prompts": N_PROMPTS,
        "smoke_n_prompts": SMOKE_N_PROMPTS,
        "optimization_n_prompts": OPTIMIZATION_N_PROMPTS,
        "batch_size": BATCH_SIZE,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "global_k_values": GLOBAL_K_VALUES,
        "role_k_values": ROLE_K_VALUES,
        "top_global_feature_pool": TOP_GLOBAL_FEATURE_POOL,
        "learned_candidate_pairs": LEARNED_CANDIDATE_PAIRS,
        "run_learned_role_mask": RUN_LEARNED_ROLE_MASK,
        "learned_lambda": LEARNED_LAMBDA,
        "learned_steps": LEARNED_STEPS,
        "learned_lr": LEARNED_LR,
        "role_names": ROLE_NAMES,
        "core_role_names": CORE_ROLE_NAMES,
        "project_root": str(PROJECT_ROOT),
        "cache_dir": str(CACHE_DIR),
        "output_dir": str(OUTPUT_DIR),
        "latest_dir": str(LATEST_DIR),
        "artifacts": {
            "stats_cache_path": str(STATS_CACHE_PATH),
            "smoke_results_csv_path": str(SMOKE_RESULTS_CSV_PATH),
            "results_csv_path": str(RESULTS_CSV_PATH),
            "learned_trace_csv_path": str(LEARNED_TRACE_CSV_PATH),
            "smoke_plot_path": str(SMOKE_PLOT_PATH),
            "plot_path": str(PLOT_PATH),
        },
        "extra": extra or {},
    }
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    return manifest

def mirror_artifacts_to_latest(paths):
    LATEST_DIR.mkdir(parents=True, exist_ok=True)
    copied = []
    for artifact_path in paths:
        artifact_path = Path(artifact_path)
        if artifact_path.exists():
            destination = LATEST_DIR / artifact_path.name
            shutil.copy2(artifact_path, destination)
            copied.append(destination)
    return copied

write_run_manifest("initialized")
print(f"Project root: {PROJECT_ROOT}")
print(f"Run version: {RUN_VERSION}")
print(f"Trial ID: {TRIAL_ID}")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Cache folder: {CACHE_DIR}")


## 3. Load Model and SAE

The v3 experiment uses the same layer 8 residual-stream SAE as v1/v2 so the curves are directly comparable.


In [ ]:
SAE_CACHE = {}

def load_sae_for_layer(layer):
    if layer in SAE_CACHE:
        return SAE_CACHE[layer]
    sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=sae_id_for_layer(layer), device=device)
    sae.eval()
    for param in sae.parameters():
        param.requires_grad_(False)
    assert sae.cfg.d_sae == EXPECTED_D_SAE
    SAE_CACHE[layer] = sae
    return sae

target_sae = load_sae_for_layer(TARGET_LAYER)
metadata = getattr(target_sae.cfg, "metadata", None)
model_kwargs = getattr(metadata, "model_from_pretrained_kwargs", None) or {}
model = HookedSAETransformer.from_pretrained_no_processing(MODEL_NAME, device=device, **model_kwargs)
model.eval()
for param in model.parameters():
    param.requires_grad_(False)

if getattr(model.tokenizer, "pad_token", None) is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = "left"
if not getattr(model.tokenizer, "is_fast", False):
    raise RuntimeError("Role labeling requires a fast tokenizer with offset mappings.")

print(f"Loaded model: {MODEL_NAME}")
print(f"Loaded SAE: {SAE_RELEASE} / {sae_id_for_layer(TARGET_LAYER)}")
print(f"SAE d_in: {target_sae.cfg.d_in}")
print(f"SAE d_sae: {target_sae.cfg.d_sae}")


## 4. Build IOI Dataset With Token Roles

Each prompt is labeled with semantic token roles using tokenizer offset mappings. The SAE hook uses these role IDs to apply different feature masks at different token roles.


In [ ]:
NAMES = [
    "John", "Mary", "Bob", "Alice", "Tom", "Sarah", "James", "Emily",
    "Robert", "Laura", "Michael", "Anna", "David", "Lisa", "Daniel", "Emma",
    "Paul", "Karen", "Mark", "Susan", "Peter", "Linda", "Kevin", "Nancy",
    "Steven", "Helen", "George", "Carol", "Brian", "Julia", "Henry", "Megan",
    "Adam", "Rachel", "Patrick", "Olivia", "Andrew", "Grace", "Edward", "Sophie",
]
PLACES = ["store", "park", "school", "office", "garden", "library", "station", "market", "museum", "theater", "church", "beach", "cafe", "hotel", "airport", "hospital"]
OBJECTS = ["book", "letter", "drink", "snack", "ticket", "phone", "gift", "photo", "bag", "card", "key", "toy", "coin", "map", "note", "pen"]
TEMPLATES = [
    "When {subject} and {io} went to the {place}, {subject} gave a {obj} to",
    "After {subject} and {io} visited the {place}, {subject} handed a {obj} to",
    "While {subject} and {io} waited near the {place}, {subject} passed a {obj} to",
    "Because {subject} and {io} were at the {place}, {subject} offered a {obj} to",
]

def build_ioi_dataset(n_prompts=1_000, seed=0):
    rng = random.Random(seed)
    records, seen, attempts = [], set(), 0
    while len(records) < n_prompts and attempts < n_prompts * 100:
        attempts += 1
        subject, io = rng.sample(NAMES, 2)
        place, obj, template = rng.choice(PLACES), rng.choice(OBJECTS), rng.choice(TEMPLATES)
        clean_prompt = template.format(subject=subject, io=io, place=place, obj=obj)
        corrupt_prompt = template.format(subject=io, io=subject, place=place, obj=obj)
        key = (clean_prompt, corrupt_prompt)
        if key in seen:
            continue
        seen.add(key)
        records.append({
            "clean_prompt": clean_prompt,
            "corrupt_prompt": corrupt_prompt,
            "answer_clean": f" {io}",
            "answer_corrupt": f" {subject}",
            "subject": subject,
            "indirect_object": io,
            "place": place,
            "object": obj,
            "template": template,
        })
    if len(records) < n_prompts:
        raise ValueError(f"Only generated {len(records)} unique prompts out of requested {n_prompts}.")
    return pd.DataFrame(records)

def token_id_for_answer(answer):
    tokens = model.to_tokens(answer, prepend_bos=False).reshape(-1)
    if tokens.numel() != 1:
        pieces = model.to_str_tokens(answer, prepend_bos=False)
        raise ValueError(f"Answer {answer!r} is not a single token: {pieces}")
    return int(tokens.item())

def find_required_span(text, value, start=0):
    index = text.find(value, start)
    if index < 0:
        raise ValueError(f"Could not find {value!r} in {text!r} after {start}")
    return index, index + len(value)

def overlap_len(a_start, a_end, b_start, b_end):
    return max(0, min(a_end, b_end) - max(a_start, b_start))

def role_ids_for_prompt(row):
    text = row["clean_prompt"]
    subject, io, place, obj = row["subject"], row["indirect_object"], row["place"], row["object"]
    subject_first = find_required_span(text, subject, 0)
    indirect_object = find_required_span(text, io, subject_first[1])
    subject_repeat = find_required_span(text, subject, indirect_object[1])
    place_span = find_required_span(text, place, indirect_object[1])
    object_span = find_required_span(text, obj, place_span[1])
    spans = [
        ("subject_first", subject_first),
        ("indirect_object", indirect_object),
        ("subject_repeat", subject_repeat),
        ("place", place_span),
        ("object", object_span),
    ]
    encoded = model.tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    token_ids, offsets = encoded["input_ids"], encoded["offset_mapping"]
    tl_tokens = model.to_tokens(text, prepend_bos=True).squeeze(0)
    if len(token_ids) + 1 != int(tl_tokens.numel()):
        raise ValueError("Tokenizer offset length does not match TransformerLens token length with BOS.")
    roles = [ROLE_TO_ID["bos"]]
    for start, end in offsets:
        best_role, best_overlap = "other", 0
        for role_name, (span_start, span_end) in spans:
            current_overlap = overlap_len(start, end, span_start, span_end)
            if current_overlap > best_overlap:
                best_overlap, best_role = current_overlap, role_name
        roles.append(ROLE_TO_ID[best_role])
    roles[-1] = ROLE_TO_ID["final_prediction"]
    return roles

def add_answer_tokens_and_roles(dataset):
    dataset = dataset.copy()
    dataset["answer_clean_id"] = [token_id_for_answer(x) for x in dataset["answer_clean"]]
    dataset["answer_corrupt_id"] = [token_id_for_answer(x) for x in dataset["answer_corrupt"]]
    dataset["role_ids"] = [role_ids_for_prompt(row) for _, row in dataset.iterrows()]
    return dataset

smoke_dataset = add_answer_tokens_and_roles(build_ioi_dataset(SMOKE_N_PROMPTS, seed=SEED))
full_dataset = add_answer_tokens_and_roles(build_ioi_dataset(N_PROMPTS, seed=SEED))
optimization_dataset = full_dataset.head(OPTIMIZATION_N_PROMPTS).copy()

print(smoke_dataset[["clean_prompt", "answer_clean", "answer_corrupt"]].head())
print(f"Smoke prompts: {len(smoke_dataset)}")
print(f"Optimization prompts: {len(optimization_dataset)}")
print(f"Full prompts: {len(full_dataset)}")
print("Example roles:", [ROLE_NAMES[x] for x in smoke_dataset.loc[0, "role_ids"]])


## 5. Role-Conditioned Intervention Helpers

The hook supports both v2-style global masks and v3 role-feature masks. `preserve_error=True` is kept from v2 so all-feature masks recover the full activation exactly.


In [ ]:
def group_tokenized_rows(dataset):
    groups = defaultdict(list)
    for index, row in dataset.reset_index(drop=True).iterrows():
        tokens = model.to_tokens(row["clean_prompt"], prepend_bos=True).squeeze(0).to(device)
        role_ids = torch.tensor(row["role_ids"], dtype=torch.long, device=device)
        if int(tokens.numel()) != int(role_ids.numel()):
            raise ValueError("Token and role lengths do not match.")
        groups[int(tokens.numel())].append((index, tokens, role_ids))
    return groups

def sae_role_mask_hook(activation, hook, sae, feature_mask=None, role_feature_mask=None, role_ids=None, preserve_error=True):
    feature_acts = sae.encode(activation)
    full_reconstruction = sae.decode(feature_acts)
    if role_feature_mask is not None:
        if role_ids is None:
            raise ValueError("role_ids are required for role_feature_mask")
        gate = role_feature_mask.to(device=feature_acts.device, dtype=feature_acts.dtype)[role_ids]
        masked_feature_acts = feature_acts * gate
    elif feature_mask is not None:
        gate = feature_mask.to(device=feature_acts.device, dtype=feature_acts.dtype).view(1, 1, -1)
        masked_feature_acts = feature_acts * gate
    else:
        masked_feature_acts = feature_acts
    masked_reconstruction = sae.decode(masked_feature_acts)
    if preserve_error:
        return masked_reconstruction + (activation - full_reconstruction)
    return masked_reconstruction

def run_logits(dataset, batch_size=BATCH_SIZE, sae=None, feature_mask=None, role_feature_mask=None, use_sae_substitution=False, preserve_error=True, detach=True):
    output_logits = [None] * len(dataset)
    groups = group_tokenized_rows(dataset)
    hook_name = hook_name_for_layer(TARGET_LAYER)
    grad_context = torch.no_grad() if detach else torch.enable_grad()
    with grad_context:
        for items in groups.values():
            for start in range(0, len(items), batch_size):
                chunk = items[start : start + batch_size]
                indices = [item[0] for item in chunk]
                tokens = torch.stack([item[1] for item in chunk], dim=0)
                role_ids = torch.stack([item[2] for item in chunk], dim=0)
                if use_sae_substitution:
                    fwd_hooks = [(hook_name, partial(sae_role_mask_hook, sae=sae, feature_mask=feature_mask, role_feature_mask=role_feature_mask, role_ids=role_ids, preserve_error=preserve_error))]
                    context = model.hooks(fwd_hooks=fwd_hooks)
                else:
                    context = nullcontext()
                with context:
                    final_logits = model(tokens)[:, -1, :]
                if detach:
                    final_logits = final_logits.detach().cpu()
                for idx, row_logits in zip(indices, final_logits):
                    output_logits[idx] = row_logits
    return torch.stack(output_logits, dim=0)

def example_logit_diffs(final_logits, dataset):
    clean_ids = torch.tensor(dataset["answer_clean_id"].to_list(), dtype=torch.long, device=final_logits.device)
    corrupt_ids = torch.tensor(dataset["answer_corrupt_id"].to_list(), dtype=torch.long, device=final_logits.device)
    row_indices = torch.arange(len(dataset), dtype=torch.long, device=final_logits.device)
    return final_logits[row_indices, clean_ids] - final_logits[row_indices, corrupt_ids]

def mean_logit_diff(final_logits, dataset):
    return example_logit_diffs(final_logits, dataset).mean()

def compute_faithfulness(sae, dataset, feature_mask=None, role_feature_mask=None, full_logit_diff=None, batch_size=BATCH_SIZE):
    if full_logit_diff is None:
        full_logits = run_logits(dataset, batch_size=batch_size, detach=True)
        full_logit_diff = mean_logit_diff(full_logits, dataset)
    masked_logits = run_logits(dataset, batch_size=batch_size, sae=sae, feature_mask=feature_mask, role_feature_mask=role_feature_mask, use_sae_substitution=True, preserve_error=True, detach=True)
    masked_logit_diff = mean_logit_diff(masked_logits, dataset)
    denominator = float(full_logit_diff.item())
    if abs(denominator) < 1e-8:
        raise ValueError(f"Full-model logit diff is too close to zero: {denominator}")
    if role_feature_mask is not None:
        active_nodes = int((role_feature_mask > 0).sum().item())
        unique_features = int(((role_feature_mask > 0).sum(dim=0) > 0).sum().item())
        mask_type = "role_feature"
    elif feature_mask is not None:
        active_nodes = int((feature_mask > 0).sum().item())
        unique_features = active_nodes
        mask_type = "global_feature"
    else:
        active_nodes = int(sae.cfg.d_sae)
        unique_features = active_nodes
        mask_type = "all_features"
    return {
        "faithfulness": float((masked_logit_diff / full_logit_diff).item()),
        "full_logit_diff": float(full_logit_diff.item()),
        "masked_logit_diff": float(masked_logit_diff.item()),
        "active_nodes": active_nodes,
        "unique_features": unique_features,
        "mask_type": mask_type,
    }

def make_global_feature_mask(top_features, k, d_sae=EXPECTED_D_SAE):
    k = min(int(k), int(d_sae))
    mask = torch.zeros(int(d_sae), device=device)
    mask[top_features[:k].to(device)] = 1.0
    return mask

def make_role_feature_mask(pair_ranking, k, n_roles=len(ROLE_NAMES), d_sae=EXPECTED_D_SAE):
    k = min(int(k), int(pair_ranking.numel()))
    flat_ids = pair_ranking[:k].to(device)
    roles = torch.div(flat_ids, d_sae, rounding_mode="floor")
    features = flat_ids % d_sae
    mask = torch.zeros((n_roles, d_sae), device=device)
    mask[roles, features] = 1.0
    return mask


## 6. Dimension and Role Check

In [ ]:
sample_tokens = model.to_tokens(smoke_dataset.loc[0, "clean_prompt"], prepend_bos=True).to(device)
_, sample_cache = model.run_with_cache(sample_tokens, names_filter=[hook_name_for_layer(TARGET_LAYER)])
sample_acts = sample_cache[hook_name_for_layer(TARGET_LAYER)]
print(f"Sample activation shape at layer {TARGET_LAYER}: {tuple(sample_acts.shape)}")
print(f"SAE d_in: {target_sae.cfg.d_in}")
print(f"SAE d_sae: {target_sae.cfg.d_sae}")
assert sample_acts.shape[-1] == target_sae.cfg.d_in
assert target_sae.cfg.d_sae == EXPECTED_D_SAE
for _, row in smoke_dataset.iterrows():
    token_count = int(model.to_tokens(row["clean_prompt"], prepend_bos=True).numel())
    assert token_count == len(row["role_ids"])
print("Role labels align with tokenized prompts.")


## 7. Cache Global and Role-Feature Statistics

In [ ]:
def cache_role_feature_stats(dataset, sae, cache_path, batch_size=BATCH_SIZE, force_recompute=False):
    cache_path = Path(cache_path)
    if cache_path.exists() and not force_recompute:
        payload = torch.load(cache_path, map_location="cpu")
        print(f"Loaded cached role-feature stats from {cache_path}")
        return {key: value.to(device) if torch.is_tensor(value) else value for key, value in payload.items()}
    d_sae, n_roles = int(sae.cfg.d_sae), len(ROLE_NAMES)
    sum_abs_all = torch.zeros(d_sae, device=device)
    sum_sq_all = torch.zeros(d_sae, device=device)
    count_all = 0
    sum_abs_by_role = torch.zeros((n_roles, d_sae), device=device)
    sum_sq_by_role = torch.zeros((n_roles, d_sae), device=device)
    count_by_role = torch.zeros(n_roles, device=device)
    groups = group_tokenized_rows(dataset)
    hook_name = hook_name_for_layer(TARGET_LAYER)
    with torch.no_grad():
        for items in tqdm(groups.values(), desc="Token length groups"):
            for start in tqdm(range(0, len(items), batch_size), leave=False, desc="Batches"):
                chunk = items[start : start + batch_size]
                tokens = torch.stack([item[1] for item in chunk], dim=0)
                role_ids = torch.stack([item[2] for item in chunk], dim=0)
                _, cache = model.run_with_cache(tokens, names_filter=[hook_name])
                feature_acts = sae.encode(cache[hook_name])
                sum_abs_all += feature_acts.abs().sum(dim=(0, 1))
                sum_sq_all += feature_acts.square().sum(dim=(0, 1))
                count_all += feature_acts.shape[0] * feature_acts.shape[1]
                for role_id in range(n_roles):
                    selected = feature_acts[role_ids == role_id]
                    if selected.numel() == 0:
                        continue
                    sum_abs_by_role[role_id] += selected.abs().sum(dim=0)
                    sum_sq_by_role[role_id] += selected.square().sum(dim=0)
                    count_by_role[role_id] += selected.shape[0]
    safe_role_counts = count_by_role.clamp_min(1).view(-1, 1)
    payload = {
        "mean_abs_all": (sum_abs_all / count_all).detach().cpu(),
        "l2_all": sum_sq_all.sqrt().detach().cpu(),
        "mean_abs_by_role": (sum_abs_by_role / safe_role_counts).detach().cpu(),
        "l2_by_role": sum_sq_by_role.sqrt().detach().cpu(),
        "count_all": int(count_all),
        "count_by_role": count_by_role.detach().cpu(),
        "role_names": ROLE_NAMES,
        "target_layer": TARGET_LAYER,
        "sae_release": SAE_RELEASE,
        "sae_id": sae_id_for_layer(TARGET_LAYER),
    }
    torch.save(payload, cache_path)
    print(f"Saved role-feature stats to {cache_path}")
    return {key: value.to(device) if torch.is_tensor(value) else value for key, value in payload.items()}

def get_decoder_feature_norm(sae):
    W_dec = sae.W_dec.detach()
    if W_dec.shape[0] == int(sae.cfg.d_sae):
        return W_dec.norm(dim=1)
    if W_dec.shape[-1] == int(sae.cfg.d_sae):
        return W_dec.norm(dim=0)
    raise ValueError(f"Cannot infer decoder feature axis from W_dec shape {tuple(W_dec.shape)}")

def build_pair_ranking(score_by_role, allowed_features=None, allowed_role_ids=None):
    scores = score_by_role.clone().to(device)
    if allowed_features is not None:
        allowed_mask = torch.zeros(scores.shape[1], dtype=torch.bool, device=device)
        allowed_mask[allowed_features.to(device)] = True
        scores[:, ~allowed_mask] = -torch.inf
    if allowed_role_ids is not None:
        role_mask = torch.zeros(scores.shape[0], dtype=torch.bool, device=device)
        role_mask[allowed_role_ids.to(device)] = True
        scores[~role_mask, :] = -torch.inf
    return scores.flatten().argsort(descending=True)

def make_rankings(stats, sae):
    decoder_norm = get_decoder_feature_norm(sae).to(device)
    global_score = stats["mean_abs_all"]
    global_ranking = global_score.argsort(descending=True)
    allowed_features = global_ranking[:TOP_GLOBAL_FEATURE_POOL]
    role_mean_abs = stats["mean_abs_by_role"]
    role_wanda = stats["l2_by_role"] * decoder_norm.view(1, -1)
    core_role_ids = CORE_ROLE_IDS.to(device)
    rankings = {
        "global_activation_mean_abs": global_ranking,
        "role_mean_abs_all_roles": build_pair_ranking(role_mean_abs, allowed_features=allowed_features),
        "role_mean_abs_core_roles": build_pair_ranking(role_mean_abs, allowed_features=allowed_features, allowed_role_ids=core_role_ids),
        "role_wanda_all_roles": build_pair_ranking(role_wanda, allowed_features=allowed_features),
        "role_wanda_core_roles": build_pair_ranking(role_wanda, allowed_features=allowed_features, allowed_role_ids=core_role_ids),
    }
    return {"global_score": global_score, "role_mean_abs": role_mean_abs, "role_wanda": role_wanda}, rankings


## 8. Pareto Evaluation Helpers

In [ ]:
def evaluate_ranked_masks(dataset, sae, rankings, global_k_values, role_k_values, batch_size=BATCH_SIZE, output_csv_path=None):
    full_logits = run_logits(dataset, batch_size=batch_size, detach=True)
    full_logit_diff = mean_logit_diff(full_logits, dataset)
    rows = []
    all_feature_mask = torch.ones(int(sae.cfg.d_sae), device=device)
    zero_feature_mask = torch.zeros(int(sae.cfg.d_sae), device=device)
    for baseline_name, feature_mask in [("all_features", all_feature_mask), ("zero_features", zero_feature_mask)]:
        metrics = compute_faithfulness(sae, dataset, feature_mask=feature_mask, full_logit_diff=full_logit_diff, batch_size=batch_size)
        rows.append({"baseline": baseline_name, "k": int(metrics["active_nodes"]), **metrics})
    for baseline_name, ranking in rankings.items():
        if baseline_name.startswith("global_"):
            for k in tqdm(global_k_values, desc=f"{baseline_name} sweep"):
                feature_mask = make_global_feature_mask(ranking, k, d_sae=int(sae.cfg.d_sae))
                metrics = compute_faithfulness(sae, dataset, feature_mask=feature_mask, full_logit_diff=full_logit_diff, batch_size=batch_size)
                rows.append({"baseline": baseline_name, "k": int(k), **metrics})
        else:
            for k in tqdm(role_k_values, desc=f"{baseline_name} sweep"):
                role_feature_mask = make_role_feature_mask(ranking, k, n_roles=len(ROLE_NAMES), d_sae=int(sae.cfg.d_sae))
                metrics = compute_faithfulness(sae, dataset, role_feature_mask=role_feature_mask, full_logit_diff=full_logit_diff, batch_size=batch_size)
                rows.append({"baseline": baseline_name, "k": int(k), **metrics})
    results = pd.DataFrame(rows)
    if output_csv_path is not None:
        results.to_csv(output_csv_path, index=False)
        print(f"Saved results to {output_csv_path}")
    return results

def plot_role_pareto(results, output_path=None, title="Role-conditioned SAE circuit Pareto"):
    fig, ax = plt.subplots(figsize=(9, 5.4))
    plot_df = results[~results["baseline"].isin(["all_features", "zero_features"])].copy()
    for baseline_name, group in plot_df.groupby("baseline"):
        ordered = group.sort_values("active_nodes")
        ax.plot(ordered["active_nodes"], ordered["faithfulness"], marker="o", linewidth=1.5, label=baseline_name)
    zero_rows = results[results["baseline"] == "zero_features"]
    if len(zero_rows):
        ax.axhline(float(zero_rows.iloc[0]["faithfulness"]), color="tab:red", linestyle="--", linewidth=1.0, label="zero features")
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=1.2, label="full model / all features")
    ax.set_xscale("log")
    ax.set_xlabel("Circuit size (active feature nodes)")
    ax.set_ylabel("Faithfulness (masked logit diff / full logit diff)")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    if output_path is not None:
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        print(f"Saved plot to {output_path}")
    plt.show()
    return fig, ax

def summarize_best(results):
    candidates = results[~results["baseline"].isin(["all_features", "zero_features"])].copy()
    best_by_k = candidates.loc[candidates.groupby("active_nodes")["faithfulness"].idxmax()].sort_values("active_nodes")
    best_by_baseline = candidates.loc[candidates.groupby("baseline")["faithfulness"].idxmax()].sort_values("faithfulness", ascending=False)
    return best_by_k, best_by_baseline


## 9. Optional Learned Role-Feature Mask

In [ ]:
def make_batches(dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True):
    groups = group_tokenized_rows(dataset)
    all_batches = []
    for items in groups.values():
        items = list(items)
        if shuffle:
            random.shuffle(items)
        for start in range(0, len(items), batch_size):
            chunk = items[start : start + batch_size]
            indices = torch.tensor([item[0] for item in chunk], dtype=torch.long)
            tokens = torch.stack([item[1] for item in chunk], dim=0)
            role_ids = torch.stack([item[2] for item in chunk], dim=0)
            all_batches.append((indices, tokens, role_ids))
    if shuffle:
        random.shuffle(all_batches)
    return all_batches

def logits_for_token_role_batch(tokens, role_ids, sae, role_feature_mask):
    hook_name = hook_name_for_layer(TARGET_LAYER)
    fwd_hooks = [(hook_name, partial(sae_role_mask_hook, sae=sae, role_feature_mask=role_feature_mask, role_ids=role_ids, preserve_error=True))]
    with model.hooks(fwd_hooks=fwd_hooks):
        return model(tokens)[:, -1, :]

def dense_role_mask_from_candidate_probs(candidate_pair_ids, probs, n_roles=len(ROLE_NAMES), d_sae=EXPECTED_D_SAE):
    dense_flat = torch.zeros(n_roles * d_sae, device=probs.device, dtype=probs.dtype)
    dense_flat = dense_flat.scatter(0, candidate_pair_ids.to(probs.device), probs)
    return dense_flat.view(n_roles, d_sae)

def train_learned_role_mask(dataset, sae, candidate_pair_ids, steps=LEARNED_STEPS, lr=LEARNED_LR, lambda_size=LEARNED_LAMBDA, batch_size=TRAIN_BATCH_SIZE):
    train_df = dataset.reset_index(drop=True).copy()
    full_logits = run_logits(train_df, batch_size=BATCH_SIZE, detach=True)
    full_diffs = example_logit_diffs(full_logits, train_df).to(device)
    clean_ids = torch.tensor(train_df["answer_clean_id"].to_list(), dtype=torch.long, device=device)
    corrupt_ids = torch.tensor(train_df["answer_corrupt_id"].to_list(), dtype=torch.long, device=device)
    candidate_pair_ids = candidate_pair_ids[:LEARNED_CANDIDATE_PAIRS].to(device)
    mask_logits = torch.nn.Parameter(torch.full((candidate_pair_ids.numel(),), 2.0, device=device))
    optimizer = torch.optim.Adam([mask_logits], lr=lr)
    trace = []
    for step in tqdm(range(steps), desc="Learning role-feature mask"):
        batches = make_batches(train_df, batch_size=batch_size, shuffle=True)
        indices, tokens, role_ids = batches[step % len(batches)]
        indices = indices.to(device)
        probs = torch.sigmoid(mask_logits)
        role_feature_mask = dense_role_mask_from_candidate_probs(candidate_pair_ids, probs)
        with torch.enable_grad():
            logits = logits_for_token_role_batch(tokens, role_ids, sae, role_feature_mask)
            row_indices = torch.arange(tokens.shape[0], device=device)
            masked_diffs = logits[row_indices, clean_ids[indices]] - logits[row_indices, corrupt_ids[indices]]
            target = full_diffs[indices].mean().detach()
            masked = masked_diffs.mean()
            reconstruction_loss = ((masked - target) / (target.abs() + 1e-6)).pow(2)
            size_loss = probs.mean()
            loss = reconstruction_loss + lambda_size * size_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        if step % 10 == 0 or step == steps - 1:
            trace.append({
                "step": int(step),
                "loss": float(loss.detach().item()),
                "reconstruction_loss": float(reconstruction_loss.detach().item()),
                "size_loss": float(size_loss.detach().item()),
                "batch_faithfulness": float((masked.detach() / target).item()),
                "mean_gate_probability": float(probs.detach().mean().item()),
                "sum_gate_probability": float(probs.detach().sum().item()),
            })
    learned_probs = torch.sigmoid(mask_logits).detach()
    learned_order = torch.argsort(learned_probs, descending=True)
    return candidate_pair_ids[learned_order].detach(), learned_probs[learned_order].detach(), pd.DataFrame(trace)


## 10. Smoke Test

In [ ]:
smoke_stats = cache_role_feature_stats(smoke_dataset, target_sae, cache_path=SMOKE_STATS_CACHE_PATH, batch_size=BATCH_SIZE, force_recompute=True)
smoke_scores, smoke_rankings = make_rankings(smoke_stats, target_sae)
smoke_results = evaluate_ranked_masks(
    smoke_dataset,
    target_sae,
    smoke_rankings,
    global_k_values=SMOKE_GLOBAL_K_VALUES,
    role_k_values=SMOKE_ROLE_K_VALUES,
    batch_size=BATCH_SIZE,
    output_csv_path=SMOKE_RESULTS_CSV_PATH,
)
write_run_manifest("smoke_completed", extra={"smoke_rows": len(smoke_results)})
smoke_results


In [ ]:
plot_role_pareto(smoke_results, output_path=SMOKE_PLOT_PATH, title="Smoke test: role-conditioned SAE circuits")


## 11. Full v3 Run

In [ ]:
feature_stats = cache_role_feature_stats(full_dataset, target_sae, cache_path=STATS_CACHE_PATH, batch_size=BATCH_SIZE, force_recompute=False)
feature_scores, feature_rankings = make_rankings(feature_stats, target_sae)
print("Top global feature ids:", feature_rankings["global_activation_mean_abs"][:10].detach().cpu().tolist())
for role_name, count in zip(ROLE_NAMES, feature_stats["count_by_role"].detach().cpu().tolist()):
    print(f"{role_name:>18}: {int(count)} token positions")


In [ ]:
learned_trace = pd.DataFrame()
if RUN_LEARNED_ROLE_MASK:
    candidate_pairs = feature_rankings["role_mean_abs_all_roles"][:LEARNED_CANDIDATE_PAIRS]
    learned_pair_ranking, learned_probs, learned_trace = train_learned_role_mask(
        optimization_dataset,
        target_sae,
        candidate_pairs,
        steps=LEARNED_STEPS,
        lr=LEARNED_LR,
        lambda_size=LEARNED_LAMBDA,
        batch_size=TRAIN_BATCH_SIZE,
    )
    feature_rankings[f"learned_role_mask_lambda_{str(LEARNED_LAMBDA).replace('.', '_')}"] = learned_pair_ranking
    learned_trace.to_csv(LEARNED_TRACE_CSV_PATH, index=False)
    print(f"Saved learned-mask trace to {LEARNED_TRACE_CSV_PATH}")
    print(learned_trace.tail())
else:
    print("RUN_LEARNED_ROLE_MASK=False, skipping learned gate optimization.")


In [ ]:
results = evaluate_ranked_masks(
    full_dataset,
    target_sae,
    feature_rankings,
    global_k_values=GLOBAL_K_VALUES,
    role_k_values=ROLE_K_VALUES,
    batch_size=BATCH_SIZE,
    output_csv_path=RESULTS_CSV_PATH,
)
best_by_k, best_by_baseline = summarize_best(results)
print("Best result per baseline:")
display(best_by_baseline[["baseline", "active_nodes", "unique_features", "faithfulness", "masked_logit_diff"]])
results


In [ ]:
plot_role_pareto(results, output_path=PLOT_PATH, title="v003: global vs role-conditioned SAE circuits")
best_rows = best_by_baseline[["baseline", "active_nodes", "unique_features", "faithfulness"]].to_dict(orient="records")
manifest = write_run_manifest("completed", extra={"rows": len(results), "best_by_baseline": best_rows, "learned_trace_rows": len(learned_trace)})
copied = mirror_artifacts_to_latest([SMOKE_RESULTS_CSV_PATH, SMOKE_PLOT_PATH, RESULTS_CSV_PATH, PLOT_PATH, LEARNED_TRACE_CSV_PATH, MANIFEST_PATH])
print("Completed run manifest:")
print(json.dumps(manifest, indent=2))
print("Mirrored latest artifacts:")
for path in copied:
    print(path)


## 12. Expected Artifacts and Reading the Result

Expected Drive layout:

```text
MyDrive/minimum_sae_circuit_discovery/
  cache/v003_role_conditioned_circuits/
    role_feature_stats_layer8_ioi.pt
  runs/v003_role_conditioned_circuits/trial_YYYYMMDD_HHMMSS/
    run_manifest.json
    smoke_role_conditioned_results.csv
    smoke_role_conditioned_pareto.png
    role_conditioned_results.csv
    role_conditioned_pareto.png
    learned_role_mask_trace.csv
  latest/v003_role_conditioned_circuits/
    run_manifest.json
    role_conditioned_results.csv
    role_conditioned_pareto.png
```

The publishable win condition is a role-conditioned curve reaching `>=0.95` faithfulness with far fewer active nodes than v2's `~500` global features. If the learned role mask wins, v4 should sweep lambda and seeds.
